# 🎬 Wav2Lip Remote Inference Server

This notebook hosts Wav2Lip on Colab's GPU and exposes an API endpoint via ngrok tunnel.

**Steps:**
1. Make sure you have selected a **GPU runtime** (Runtime → Change runtime type → T4 GPU)
2. Run **Cell 1** to install everything
3. Set your ngrok auth token in **Cell 2** and run it
4. Copy the printed URL into your local `.env` as `WAV2LIP_REMOTE_URL=<url>`
5. Run your local pipeline — it will send inference to this Colab!

> Get a free ngrok auth token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ============================================================
# CELL 1: Install dependencies and download Wav2Lip
# ============================================================

# 1a. Clone Wav2Lip
!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip

# 1b. Download model weights
!mkdir -p /content/Wav2Lip/checkpoints
!wget -q --show-progress -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth \
    "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth?download=true"

# 1c. Download s3fd face detector weights
!wget -q --show-progress -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth \
    "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/s3fd-619a316812.pth"

# 1d. Install Python deps
!pip install -q flask pyngrok librosa==0.10.2

# 1e. Patch Wav2Lip for modern PyTorch / librosa / numpy
import pathlib

def patch_file(path, replacements):
    text = pathlib.Path(path).read_text()
    for old, new in replacements:
        text = text.replace(old, new, 1)
    pathlib.Path(path).write_text(text)

# audio.py: librosa.filters.mel needs keyword sr=
patch_file("/content/Wav2Lip/audio.py", [
    ("return librosa.filters.mel(hp.sample_rate, hp.n_fft,",
     "return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,"),
])

# inference.py: torch.load needs weights_only=False
patch_file("/content/Wav2Lip/inference.py", [
    ("\t\tcheckpoint = torch.load(checkpoint_path)\n",
     "\t\tcheckpoint = torch.load(checkpoint_path, weights_only=False)\n"),
    ("\t\tcheckpoint = torch.load(checkpoint_path,\n"
     "\t\t\t\t\t\t\t\tmap_location=lambda storage, loc: storage)\n",
     "\t\tcheckpoint = torch.load(checkpoint_path,\n"
     "\t\t\t\t\t\t\t\tmap_location=lambda storage, loc: storage,\n"
     "\t\t\t\t\t\t\t\tweights_only=False)\n"),
])

# sfd_detector.py: torch.load needs weights_only=False
sfd_path = "/content/Wav2Lip/face_detection/detection/sfd/sfd_detector.py"
patch_file(sfd_path, [
    ("import os\nimport cv2\nfrom torch.utils.model_zoo import load_url",
     "import os\nimport cv2\nimport torch\nfrom torch.utils.model_zoo import load_url"),
    ("            model_weights = torch.load(path_to_detector)",
     "            model_weights = torch.load(path_to_detector, weights_only=False)"),
])

# Create temp directory
!mkdir -p /content/Wav2Lip/temp

import torch
print(f"\n✅ Wav2Lip setup complete!")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# CELL 2: Start the inference server with ngrok tunnel
# ============================================================

# ⚠️  SET YOUR NGROK AUTH TOKEN BELOW
# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # <--- PASTE YOUR TOKEN HERE

import os, sys, uuid, subprocess, shutil, threading, time, gc
from pathlib import Path

# Add Wav2Lip to path
sys.path.insert(0, "/content/Wav2Lip")
os.chdir("/content/Wav2Lip")

from flask import Flask, request, send_file, jsonify
from pyngrok import ngrok, conf

# ── Pre-load model once ──────────────────────────────────────
import torch
import numpy as np
import cv2
import audio
import face_detection
from models import Wav2Lip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[server] Using device: {device}")

def _load_wav2lip_model(checkpoint_path):
    model = Wav2Lip()
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    s = ckpt["state_dict"]
    new_s = {k.replace("module.", ""): v for k, v in s.items()}
    model.load_state_dict(new_s)
    del ckpt, s, new_s
    gc.collect()
    torch.cuda.empty_cache()
    return model.to(device).eval()

MODEL_PATH = "/content/Wav2Lip/checkpoints/wav2lip_gan.pth"
wav2lip_model = _load_wav2lip_model(MODEL_PATH)
print("[server] Wav2Lip model loaded on GPU ✅")

# ── Inference helpers ─────────────────────────────────────────
mel_step_size = 16

def _face_detect(images, pads, nosmooth, batch_size=16):
    detector = face_detection.FaceAlignment(
        face_detection.LandmarksType._2D, flip_input=False, device=device
    )
    while True:
        predictions = []
        try:
            for i in range(0, len(images), batch_size):
                predictions.extend(
                    detector.get_detections_for_batch(np.array(images[i : i + batch_size]))
                )
        except RuntimeError:
            if batch_size == 1:
                raise
            batch_size //= 2
            continue
        break
    results = []
    pady1, pady2, padx1, padx2 = pads
    for rect, image in zip(predictions, images):
        if rect is None:
            raise ValueError("Face not detected in one or more frames.")
        y1 = max(0, rect[1] - pady1)
        y2 = min(image.shape[0], rect[3] + pady2)
        x1 = max(0, rect[0] - padx1)
        x2 = min(image.shape[1], rect[2] + padx2)
        results.append([x1, y1, x2, y2])
    boxes = np.array(results)
    if not nosmooth:
        for i in range(len(boxes)):
            T = 5
            if i + T > len(boxes):
                window = boxes[len(boxes) - T:]
            else:
                window = boxes[i : i + T]
            boxes[i] = np.mean(window, axis=0)
    results = [
        [image[y1:y2, x1:x2], (y1, y2, x1, x2)]
        for image, (x1, y1, x2, y2) in zip(images, boxes)
    ]
    del detector
    gc.collect()
    torch.cuda.empty_cache()
    return results


def run_inference(face_path, audio_path, outfile, pads=(0, 10, 0, 0),
                  resize_factor=1, wav2lip_batch_size=128,
                  face_det_batch_size=16, nosmooth=False):
    img_size = 96

    # ── Read frames ──
    if face_path.lower().endswith((".jpg", ".png", ".jpeg")):
        full_frames = [cv2.imread(face_path)]
        fps = 25.0
        static = True
    else:
        cap = cv2.VideoCapture(face_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        full_frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if resize_factor > 1:
                frame = cv2.resize(
                    frame,
                    (frame.shape[1] // resize_factor, frame.shape[0] // resize_factor),
                )
            full_frames.append(frame)
        cap.release()
        static = False

    print(f"[inference] {len(full_frames)} frames @ {fps:.1f} fps")

    # ── Mel spectrogram ──
    wav = audio.load_wav(audio_path, 16000)
    mel = audio.melspectrogram(wav)
    mel_chunks = []
    mel_idx_multiplier = 80.0 / fps
    i = 0
    while True:
        start_idx = int(i * mel_idx_multiplier)
        if start_idx + mel_step_size > len(mel[0]):
            mel_chunks.append(mel[:, len(mel[0]) - mel_step_size :])
            break
        mel_chunks.append(mel[:, start_idx : start_idx + mel_step_size])
        i += 1

    full_frames = full_frames[: len(mel_chunks)]

    # ── Face detection ──
    if static:
        face_det_results = _face_detect([full_frames[0]], pads, nosmooth, face_det_batch_size)
    else:
        face_det_results = _face_detect(full_frames, pads, nosmooth, face_det_batch_size)

    # ── Batched inference ──
    frame_h, frame_w = full_frames[0].shape[:2]
    temp_avi = "/content/Wav2Lip/temp/result.avi"
    out = cv2.VideoWriter(
        temp_avi,
        cv2.VideoWriter_fourcc(*"DIVX"),
        fps,
        (frame_w, frame_h),
    )

    img_batch, mel_batch, frame_batch, coords_batch = [], [], [], []
    for idx, m in enumerate(mel_chunks):
        fidx = 0 if static else idx % len(full_frames)
        frame_to_save = full_frames[fidx].copy()
        face, coords = face_det_results[fidx if not static else 0].copy()
        face = cv2.resize(face, (img_size, img_size))
        img_batch.append(face)
        mel_batch.append(m)
        frame_batch.append(frame_to_save)
        coords_batch.append(coords)

        if len(img_batch) >= wav2lip_batch_size or idx == len(mel_chunks) - 1:
            ib = np.asarray(img_batch)
            mb = np.asarray(mel_batch)
            ib_masked = ib.copy()
            ib_masked[:, img_size // 2 :] = 0
            ib = np.concatenate((ib_masked, ib), axis=3) / 255.0
            mb = mb.reshape(len(mb), mb.shape[1], mb.shape[2], 1)

            ib_t = torch.FloatTensor(ib.transpose(0, 3, 1, 2)).to(device)
            mb_t = torch.FloatTensor(mb.transpose(0, 3, 1, 2)).to(device)

            with torch.no_grad():
                pred = wav2lip_model(mb_t, ib_t)

            pred = pred.cpu().numpy().transpose(0, 2, 3, 1) * 255.0
            del ib_t, mb_t

            for p, f, c in zip(pred, frame_batch, coords_batch):
                y1, y2, x1, x2 = c
                p = cv2.resize(p.astype(np.uint8), (x2 - x1, y2 - y1))
                f[y1:y2, x1:x2] = p
                out.write(f)

            img_batch, mel_batch, frame_batch, coords_batch = [], [], [], []

    out.release()
    torch.cuda.empty_cache()

    # ── Mux audio ──
    cmd = f'ffmpeg -y -i "{audio_path}" -i {temp_avi} -strict -2 -q:v 1 "{outfile}"'
    subprocess.call(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    return os.path.exists(outfile)


# ── Flask App ─────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    used = torch.cuda.memory_allocated(0) / 1024**3
    return jsonify({
        "status": "ok",
        "device": device,
        "gpu": torch.cuda.get_device_name(0),
        "vram_total_gb": round(mem, 1),
        "vram_used_gb": round(used, 2),
        "model": "wav2lip_gan"
    })

@app.route("/lip_sync", methods=["POST"])
def lip_sync():
    job_id = uuid.uuid4().hex[:8]
    work_dir = Path(f"/content/jobs/{job_id}")
    work_dir.mkdir(parents=True, exist_ok=True)
    try:
        face_file = request.files.get("face")
        audio_file = request.files.get("audio")
        if not face_file or not audio_file:
            return jsonify({"error": "Both 'face' and 'audio' files required."}), 400

        face_path = str(work_dir / face_file.filename)
        audio_path = str(work_dir / audio_file.filename)
        out_path = str(work_dir / "output.mp4")

        face_file.save(face_path)
        audio_file.save(audio_path)

        pads = [int(x) for x in request.form.get("pads", "0,10,0,0").split(",")]
        resize_factor = int(request.form.get("resize_factor", "1"))
        wav2lip_batch_size = int(request.form.get("wav2lip_batch_size", "128"))
        face_det_batch_size = int(request.form.get("face_det_batch_size", "16"))
        nosmooth = request.form.get("nosmooth", "0") in {"1", "true", "yes"}

        t0 = time.time()
        print(f"[job {job_id}] Starting inference...")

        ok = run_inference(
            face_path, audio_path, out_path,
            pads=pads,
            resize_factor=resize_factor,
            wav2lip_batch_size=wav2lip_batch_size,
            face_det_batch_size=face_det_batch_size,
            nosmooth=nosmooth,
        )

        elapsed = time.time() - t0
        print(f"[job {job_id}] Done in {elapsed:.1f}s")

        if not ok:
            return jsonify({"error": "Wav2Lip produced no output."}), 500

        return send_file(out_path, mimetype="video/mp4", as_attachment=True,
                         download_name=f"lip_synced_{job_id}.mp4")

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500
    finally:
        def _cleanup():
            time.sleep(5)
            shutil.rmtree(work_dir, ignore_errors=True)
        threading.Thread(target=_cleanup, daemon=True).start()


# ── Start ngrok tunnel and Flask ──────────────────────────────
if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
else:
    print("⚠️  No NGROK_AUTH_TOKEN set! Tunnel may be rate-limited.")
    print("   Get a free token: https://dashboard.ngrok.com/get-started/your-authtoken")

public_url = ngrok.connect(5000, "http").public_url

print()
print("=" * 60)
print("🚀 Wav2Lip Remote Server is LIVE!")
print(f"   Public URL: {public_url}")
print()
print("   Copy this into your local .env:")
print(f"   WAV2LIP_REMOTE_URL={public_url}")
print("=" * 60)
print()

app.run(port=5000)